## AUTOMATIZACION ANALISIS SISMICO     -      09 - ANALISIS SISMICO SEGUN NSR-10

In [44]:
%pip install comtypes
%pip install xlwings

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Condiciones de uso:
- El programa de Etabs debe estar corrido y abierto, y el Excel debe estar abierto.
- Tener en cuenta la celda de referencia, esta es la primera fila de la masa de piso (cubierta generalmente)
- De preferencia no correr la linea que corre nuevamente el modelo, si no, correr el modelo manual para ahorrar tiempo.

Librerias 

In [45]:
import comtypes.client
import xlwings as xw
import pandas as pd
import numpy as np

#print(comtypes.__version__)

Conectamos el modelo de etabs en el que trabajaremos

In [46]:
# conectarse a la instancia activa de etabs
helper = comtypes.client.CreateObject('ETABSv1.Helper')  # Creamos un objeto para conectarnos a etabs, 'ETABSv1.Helper'es el ayudando de etabs para esta tarea
programa_abierto= helper.GetObject("CSI.ETABS.API.ETABSObject") # busca el programa de etabs que este abierto

if programa_abierto is None:
    raise RuntimeError("No se encontró ninguna instancia de ETABS abierta. Verifica que ETABS esté corriendo con un modelo cargado.")
etabs_model= programa_abierto.SapModel   # extrae la propiedad mas importante del objeto de etabs.
etabs_model.SetPresentUnits(6)
print("conectado a:", etabs_model.GetModelFilename())
print(programa_abierto)

conectado a: C:\Users\andre\Documents\ATRES_INGENIERIA\TORRRE APARTAMENTE HACIENDA CANNAN\02-MODELO\V05.MODELO HACIENDA CANAAN_pergola_2.EDB
<POINTER(cOAPI) ptr=0x202868bf8a0 at 2028c18fb50>


Conectamos con el libro de excel que quieremos trabajar

In [47]:
# Conectarte por nombre exacto del archivo (con o sin extensión, ambos funcionan)
excel_libro = xw.Book('V01.ANALISIS SISMICO SEGUN NSR-10 PERGOLA_2'
'.xlsx') # abre y conecta el libro de excel
# Seleccionar la hoja donde vas a trabajar
hoja_excel = 'Análisis' # Nombre de la hoja donde se hace el proceso
hoja = excel_libro.sheets[hoja_excel]  
fila_inicio = 210 # fila donde se empiezan a pegar los datos

EXTRAEMOS MASA DE PISO E INSERTAMOS EN EXCEL. Se pregunta si ya esta pegado o no. 

In [48]:
Masa_tabla = "Mass Summary by Story" # Nombre de la tabla donde se extrae la info

#Extraemos la tabla completa
version_tabla_masa = 0 # version de la tabla 
nombres_colums_masa = [] # guardamos el nombre de las columnas
num_filas_masa = 0 # guardamos el numero de filas
TableData_masa = [] # informacion completa de la tabla masa de etabs
 
#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, version_tabla_masa, nombres_colums_masa, num_filas_masa, TableData_masa, ret = etabs_model.DatabaseTables.GetTableForDisplayArray( Masa_tabla, nombres_colums_masa,"", version_tabla_masa, nombres_colums_masa, num_filas_masa, TableData_masa   )
#print(nombres_colums_masa)   # nombres de columnas            #################### PODRIA SOLO IMPORTAR OUTCASE Y UX ####################
#print(num_filas)            # cuántas filas trajo


n_cols = len(nombres_colums_masa) # cantidad de columnas extraidas de la tabla
array_masa = np.array(TableData_masa).reshape(-1, n_cols)  # convertimos la TableData_masa en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
masa_tabla = pd.DataFrame(array_masa, columns=nombres_colums_masa) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

#masa_tabla.head() # muestra una parte de la tabla.

# Extramos la masa por piso

masa_final = masa_tabla[['Story', 'UX']].copy()
masa_final['UX'] = masa_final['UX'].astype(float).round(3)

nombres_stories = masa_final['Story']

num_stories = len(masa_final) # cantidad de pisos
#print("Cantidad de stories:", num_stories)
masa_final

,Story,UX
0,cercha 2,0.078
1,Nivel 3 N+5.04,0.255
2,Base N+0.00,0.000


In [49]:
# Pegamos la masa en la hoja de excel


# Leemos las columnas A y B del rango que ocuparía la nueva información
df_etabs = pd.DataFrame(masa_final.values).reset_index(drop=True) # preparamos la información de etabs para comparar.

masa_actual = hoja.range(f'A{fila_inicio}:B{fila_inicio + num_stories - 1}').value
df_excel_act = pd.DataFrame(masa_actual).reset_index(drop=True) # transformamos los datos de excel en un pandas para comparar


def actualizar_masa(fila_inicio,tabla_masa, col_story='A'): # (Analisís,210,masa_final,'B')
    n_necesarias = len(tabla_masa) # cuantas filas tiene la tabla nueva 

    fila = fila_inicio # 
    while (hoja.range(f'A{fila}').value != 'M ='): # tomo de criterio para insertar celdas la celda " M ="
        fila += 1            # cuento cuantas celdas hay desde el primera hasta esta "M ="
        if fila == 240:
            print("fallo la impresion de masa - revisar celda (M =)")
            break
    n_actuales = fila-fila_inicio-1 # Cuantas celdas hay actualmente para insertar la masa

    diferencia = n_necesarias-n_actuales
    
    if diferencia > 0:
        #Faltan filas ->> insertamos las que faltan
        fila_insercion = fila_inicio 
        hoja.range(f'{fila_insercion+1}:{fila_insercion+diferencia}').insert(shift='down')

        hoja.range(f'{fila_inicio}:{fila_inicio}').copy()

        # Pega SOLO los formatos en la nueva fila (mantiene el cajón gris, bordes y [ton])
        hoja.range(f'{fila_insercion}:{fila_insercion+diferencia-1}').paste(paste='formats')
        print(f"Se insertaron {diferencia} fila(s)")

    elif diferencia < 0:
        diferencia = abs(diferencia)
        hoja.range(f'{fila_inicio+1}:{fila_inicio+diferencia}').delete(shift='up') #borramos las celdas

        print("Se recomienda borrar celdas")

    else:
        print("El número de filas ya coincide, no hace falta insertar/eliminar")
    
    hoja.range(f'{col_story}{fila_inicio}').options(pd.DataFrame, index=False, header=False).value = tabla_masa

actualizar_masa(fila_inicio,masa_final, col_story='A')


hoja.range(f'B{fila_inicio + num_stories +1 }').value = f"=SUM(B{fila_inicio}:B{fila_inicio + num_stories})" # garantizo que la suma de la masa se conserve
hoja.range(f'B{fila_inicio + num_stories + 6}').value = f"=B{fila_inicio + num_stories-1}" # garantizo que la celda ( -m )

El número de filas ya coincide, no hace falta insertar/eliminar


EXTRAEMOS LAS FHEX,Y & SPECX,Y Y LAS PEGAMOS EN EL EXCEL.  


In [50]:
BR_tabla = "Base Reactions" # Nombre de la tabla donde se extrae la info

#Extraemos la tabla completa
version_tabla_BR = 0 # version de la tabla 
nombres_colums_BR = [] # guardamos el nombre de las columnas
num_filas_BR = 0 # guardamos el numero de filas
TableData_BR = [] # informacion completa de la tabla BR de etabs

etabs_model.DatabaseTables.SetLoadCasesSelectedForDisplay(['FHEX','FHEY','SPECX','SPECY','ELASX','ELASY'])

#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, version_tabla_BR, nombres_colums_BR, num_filas_BR, TableData_BR, ret = etabs_model.DatabaseTables.GetTableForDisplayArray( BR_tabla, nombres_colums_BR,"", version_tabla_BR, nombres_colums_BR, num_filas_BR, TableData_BR   )
#print(nombres_colums_BR)   # nombres de columnas            #################### PODRIA SOLO IMPORTAR OUTCASE Y FX, FY ####################
#print(num_filas)            # cuántas filas trajo


n_cols = len(nombres_colums_BR) # cantidad de columnas extraidas de la tabla
array_BR = np.array(TableData_BR).reshape(-1, n_cols)  # convertimos la TableData_BR en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
BR_tabla = pd.DataFrame(array_BR, columns=nombres_colums_BR) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

BR_tabla.head() # muestra una parte de la tabla.


# Extramos las FX & FY

df_BR = BR_tabla[['OutputCase','FX','FY']].copy()

# Filtrar la fila del load case que te interesa
fila_FHEX = df_BR[df_BR['OutputCase'] == 'FHEX']  # df_BR['OutputCase'] == 'FHEX' -> busca donde hay registro de FHEX en OutputCase...
fila_FHEY = df_BR[df_BR['OutputCase'] == 'FHEY']  # fila_FHEX = df_BR[df_BR... --> Extrae la fila de informacion completa donde hay la coincidencia. 

fila_SPECX = df_BR[df_BR['OutputCase'] == 'SPECX']  # busca donde hay registro de SPECX en OutputCase.
fila_SPECY = df_BR[df_BR['OutputCase'] == 'SPECY']  # busca donde hay registro de SPECY en OutputCase.

if len(fila_FHEX) == 0:
    raise ValueError("No se encontró el load case 'FHEX' en la tabla Base Reactions")
if len(fila_FHEY) == 0:
    raise ValueError("No se encontró el load case 'FHEY' en la tabla Base Reactions")

# Extraer las 4 celdas puntuales

FHE_X = fila_FHEX[['FX','FY']].astype(float).round(4)
FHE_Y = fila_FHEY[['FX','FY']].astype(float).round(4)
df_FHE_final = pd.concat([FHE_X, FHE_Y], ignore_index=True) #unimos los pandas para insertar uno solo

if len(fila_SPECX) == 0:
    raise ValueError("No se encontró el load case 'SPECX' en la tabla Base Reactions")
if len(fila_SPECY) == 0:
    raise ValueError("No se encontró el load case 'SPECY' en la tabla Base Reactions")


SPECX = fila_SPECX[['FX','FY']].astype(float).round(4)
SPECY = fila_SPECY[['FX','FY']].astype(float).round(4)
df_SPEC_final = pd.concat([SPECX, SPECY], ignore_index=True) #unimos los pandas para insertar uno solo

#Escribimos los datos de las FHE en la hoja de excel.
hoja.range(f'B{fila_inicio + num_stories+ 82}').value = df_FHE_final.values
hoja.range(f'B{fila_inicio + num_stories+ 308}').options(pd.DataFrame, index=False, header=False).value = df_SPEC_final


BR_tabla.head()

,OutputCase,CaseType,StepType,StepNumber,StepLabel,FX,FY,FZ,MX,MY,MZ,X,Y,Z
0,ELASX,LinRespSpec,Max,None,None,2.047,1.6139,0,12.4321,15.8229,51.9676,0,0,0
1,ELASY,LinRespSpec,Max,None,None,0.6369,2.4275,0,18.6932,4.9524,78.6748,0,0,0
2,SPECX,LinRespSpec,Max,None,None,0.9311,0.7336,0,5.6513,7.1973,23.623,0,0,0
3,SPECY,LinRespSpec,Max,None,None,0.5169,1.9709,0,15.177,4.0197,63.8757,0,0,0
4,FHEX,LinStatic,None,None,None,-2.6189,0,0,-0.0008,-20.1942,-6.473,0,0,0


# EXTRACCION DE LOS DATOS DE MODAL PARTICIPATING MASS RATIO. LOS MAXIMOS Y EL CHEQUEO DE CUMPLIMIENTO.

In [51]:
#hoja.range(f'A{fila_inicio+num_stories+20}:F{fila_inicio+num_stories+22}')

Modal_tabla = "Modal Participating Mass Ratios" # Nombre de la tabla donde se extrae la info

#Extraemos la tabla completa
version_tabla_modal = 0 # version de la tabla 
nombres_colums_modal = [] # guardamos el nombre de las columnas
num_filas_modal= 0 # guardamos el numero de filas
TableData_modal = [] # informacion completa de la tabla modal de etabs
 
#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, version_tabla_modal, nombres_colums_modal, num_filas_modal, TableData_modal, ret = etabs_model.DatabaseTables.GetTableForDisplayArray( Modal_tabla, nombres_colums_modal,"", version_tabla_modal, nombres_colums_modal, num_filas_modal, TableData_modal   )
#print(nombres_colums_modal)   # nombres de columnas            #################### PODRIA SOLO IMPORTAR OUTCASE Y UX ####################


n_cols = len(nombres_colums_modal) # cantidad de columnas extraidas de la tabla
array_modal = np.array(TableData_modal).reshape(-1, n_cols)  # convertimos la TableData_modal en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
modal_tabla = pd.DataFrame(array_modal, columns=nombres_colums_modal) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

#modal_tabla.head() # muestra una parte de la tabla.

# Extramos la masa por piso
df_modal = modal_tabla[['Case', 'Mode','Period','UX','UY','RZ']].copy() #extraigo lo que necesito 


df_modal['UX'] = df_modal['UX'].astype(float) # convertimos de string a flotante
df_modal['UY'] = df_modal['UY'].astype(float)
df_modal['RZ'] = df_modal['RZ'].astype(float)

fila_modal1 = df_modal.loc[[df_modal['UX'].idxmax()]] # buscamos el mayor en cada caso
fila_modal2 = df_modal.loc[[df_modal['UY'].idxmax()]]
fila_modal3 = df_modal.loc[[df_modal['RZ'].idxmax()]]

df_modal_participacion = pd.concat([fila_modal1, fila_modal2, fila_modal3], ignore_index=True) #unimos los pandas para insertar uno solo
#insertamos el pandas en la hoja de excel.
hoja.range(f'A{fila_inicio+num_stories+20}:F{fila_inicio+num_stories+22}').options(pd.DataFrame, index=False, header=False).value = df_modal_participacion # pego la tabla de modal


# REALIAMOS EL CHEQUEO DEL CUMPLIMIENTO DE LA SUFICIENTE EXCITACION DE MASA

df_modal_chequeo = modal_tabla[['Mode','Period','SumUX','SumUY']].copy() #extraigo lo que necesito 
df_modal_chequeo['SumUX'] = df_modal_chequeo['SumUX'].astype(float) # convertimos de string a flotante
df_modal_chequeo['SumUY'] = df_modal_chequeo['SumUY'].astype(float)

df_modal_ultimas = df_modal_chequeo.tail(5)

hoja.range(f'B{fila_inicio+num_stories+286}:E{fila_inicio+num_stories+296}').options(pd.DataFrame, index=False, header=False).value = df_modal_ultimas # pego la tabla de modal
#print(df_modal_participacion)


AJUSTE SISMICO - SCALE FACTORS U1 y U2 PARA ELAS Y DIS

In [52]:
#creamos una funcion para que realice el proceso para cada loas case.
def actualizar_scale_factor(nombre_case, U_a_modificar):  # U a modificar --> {'U1': LEIDO EXCEL, 'U2': LEIDO EXCEL}

    NumberLoads, LoadName, Func, SF, CSys, Ang = 0, [], [], [], [], []
    [NumberLoads, LoadName, Func, SF, CSys, Ang, ret] = etabs_model.LoadCases.ResponseSpectrum.GetLoads( nombre_case, NumberLoads, LoadName, Func, SF, CSys, Ang )

    # Reconstruimos las listas como Python puro, forzando el tipo correcto
    LoadName = [str(x) for x in LoadName]
    Func = [str(x) for x in Func]
    SF = [float(x) for x in SF]
    CSys = [str(x) for x in CSys]
    Ang = [float(x) for x in Ang]

    for i, nombre in enumerate(LoadName): #recorremos la lista de load cases hasta que el que vayamos a cambiar sea encontrado
        if nombre in U_a_modificar: # pregundamos si alguno coincide
            SF[i] = float(U_a_modificar[nombre])  # si uno coincide modificamos el scale factor que le corresponde con el que ingresamos.

    
    #for nombre_lista, lista in [('LoadName', LoadName), ('Func', Func), ('SF', SF), ('CSys', CSys), ('Ang', Ang)]:
     #   print(nombre_lista, [type(x) for x in lista])



    resultado = etabs_model.LoadCases.ResponseSpectrum.SetLoads( nombre_case, NumberLoads, LoadName, Func, SF, CSys, Ang  )
    ret = resultado[-1]
    return ret == 0

U1_ELASX= float(hoja.range(f'B{fila_inicio+num_stories+366}').value)  # EXTRAMOS LOS SCALE FACTOR DE ELAS X  
U2_ELASX = float(hoja.range(f'C{fila_inicio+num_stories+366}').value)


U1_ELASY= float(hoja.range(f'B{fila_inicio+num_stories+367}').value)  # EXTRAMOS LOS SCALE FACTOR DE ELAS Y  
U2_ELASY = float(hoja.range(f'C{fila_inicio+num_stories+367}').value)
                 

U1_DISX= float(hoja.range(f'B{fila_inicio+num_stories+369}').value)  # EXTRAMOS LOS SCALE FACTOR DE DIS X  
U2_DISX = float(hoja.range(f'C{fila_inicio+num_stories+369}').value)

U1_DISY= float(hoja.range(f'B{fila_inicio+num_stories+370}').value)  # EXTRAMOS LOS SCALE FACTOR DE DIS Y  
U2_DISY = float(hoja.range(f'C{fila_inicio+num_stories+370}').value)

if etabs_model.GetModelIsLocked():  # Preguntamos si esta bloqueado el modelo
    etabs_model.SetModelIsLocked(False) # Desbloquea el modelo

ok = actualizar_scale_factor('ELASX',{'U1': U1_ELASX, 'U2': U2_ELASX})
print("Actualizado correctamente" if ok else "Falló la actualización")
ok = actualizar_scale_factor('ELASY',{'U1': U1_ELASY, 'U2': U2_ELASY})
print("Actualizado correctamente" if ok else "Falló la actualización")
ok = actualizar_scale_factor('DISX' ,{'U1': U1_DISX , 'U2': U2_DISX})
print("Actualizado correctamente" if ok else "Falló la actualización")
ok = actualizar_scale_factor('DISY' ,{'U1': U1_DISY , 'U2': U2_DISY})
print("Actualizado correctamente" if ok else "Falló la actualización")

Actualizado correctamente
Actualizado correctamente
Actualizado correctamente
Actualizado correctamente


# Corremos el programa de nuevo - ETABS

In [53]:
# Desbloqueamos el modelo por si quedó bloqueado de una corrida anterior
#etabs_model.SetModelIsLocked(False)

# Corremos el análisis de nuevo
ret_analisis = etabs_model.Analyze.RunAnalysis()

if ret_analisis == 0:
    print("Análisis corrido correctamente")
else:
    print("Hubo un problema al correr el análisis")

Análisis corrido correctamente


# CHEQUEO DE AJUSTE SISMICO - EXTRAEMOS ELAS X & Y

In [54]:
BR_tabla1 = "Base Reactions" # Nombre de la tabla donde se extrae la info

#Extraemos la tabla completa
version_tabla_BR1 = 0 # version de la tabla 
nombres_colums_BR1 = [] # guardamos el nombre de las columnas
num_filas_BR1 = 0 # guardamos el numero de filas
TableData_BR1 = [] # informacion completa de la tabla BR de etabs
 
#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, version_tabla_BR1, nombres_colums_BR1, num_filas_BR1, TableData_BR1, ret = etabs_model.DatabaseTables.GetTableForDisplayArray( BR_tabla1, nombres_colums_BR1,"", version_tabla_BR1, nombres_colums_BR1, num_filas_BR1, TableData_BR1 )
#print(nombres_colums_BR)   # nombres de columnas            #################### PODRIA SOLO IMPORTAR OUTCASE Y FX, FY ####################
#print(num_filas)            # cuántas filas trajo


n_cols = len(nombres_colums_BR1) # cantidad de columnas extraidas de la tabla
array_BR1 = np.array(TableData_BR1).reshape(-1, n_cols)  # convertimos la TableData_BR en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
BR_tabla1 = pd.DataFrame(array_BR1, columns=nombres_colums_BR1) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

BR_tabla1.head() # muestra una parte de la tabla.


# Extramos las FX & FY

df_BR1 = BR_tabla1[['OutputCase','FX','FY']].copy()

# Filtrar la fila del load case que te interesa
fila_ELASX= df_BR1[df_BR1['OutputCase'] == 'ELASX']  # df_BR['OutputCase'] == 'FHEX' -> busca donde hay registro de FHEX en OutputCase...
fila_ELASY = df_BR1[df_BR1['OutputCase'] == 'ELASY']  # fila_FHEX = df_BR[df_BR... --> Extrae la fila de informacion completa donde hay la coincidencia. 



if len(fila_ELASX) == 0:
    raise ValueError("No se encontró el load case 'FHEX' en la tabla Base Reactions")
if len(fila_ELASY) == 0:
    raise ValueError("No se encontró el load case 'FHEY' en la tabla Base Reactions")

# Extraer las 4 celdas puntuales

ELAS_X = fila_ELASX[['FX','FY']].astype(float)
ELAS_Y = fila_ELASY[['FX','FY']].astype(float)
df_ELAS_final = pd.concat([ELAS_X, ELAS_Y], ignore_index=True) #unimos los pandas para insertar uno solo

#Escribimos los datos de las FHE en la hoja de excel.
hoja.range(f'B{fila_inicio + num_stories+ 373}').options(pd.DataFrame, index=False, header=False).value = df_ELAS_final



# EXTRACCION DE LAS MAXIMAS DERIVAS DE PISO - PARA LA COMBINACIONES DE CARGA _DER

In [55]:
Drifts_tabla = "Story Drifts" # Nombre de la tabla donde se extrae la info

#Extraemos la tabla completa
version_tabla_drifts = 0 # version de la tabla 
nombres_colums_drifts = [] # guardamos el nombre de las columnas
num_filas_drifts= 0 # guardamos el numero de filas
TableData_drifts= [] # informacion completa de la tabla drifts de etabs

etabs_model.DatabaseTables.SetLoadCombinationsSelectedForDisplay(["U_B.2.4-5_1 DER_H","U_B.2.4-5_1 DER","U_B.2.4-5_2 DER_H","U_B.2.4-5_2 DER","U_B.2.4-5_3 DER_H",
    "U_B.2.4-5_3 DER","U_B.2.4-5_4 DER_H","U_B.2.4-5_4 DER","U_B.2.4-7_1 DER_H","U_B.2.4-7_1 DER","U_B.2.4-7_2 DER_H","U_B.2.4-7_2 DER","U_B.2.4-7_3 DER_H","U_B.2.4-7_3 DER",
"U_B.2.4-7_4 DER_H","U_B.2.4-7_4 DER"])

etabs_model.DatabaseTables.SetLoadCasesSelectedForDisplay([])

#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, version_tabla_drifts, nombres_colums_drifts, num_filas_drifts, TableData_drifts, ret = etabs_model.DatabaseTables.GetTableForDisplayArray( Drifts_tabla, nombres_colums_drifts,"", version_tabla_drifts, nombres_colums_drifts, num_filas_drifts, TableData_drifts   )
#print(nombres_colums_modal)   # nombres de columnas            #################### PODRIA SOLO IMPORTAR OUTCASE Y UX ####################


n_cols = len(nombres_colums_drifts) # cantidad de columnas extraidas de la tabla
array_drifts = np.array(TableData_drifts).reshape(-1, n_cols)  # convertimos la TableData_modal en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
drifts_tabla = pd.DataFrame(array_drifts, columns=nombres_colums_drifts) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

drifts_tabla.head() # muestra una parte de la tabla.

## Extramos la masa por piso
#df_drifts = drifts_tabla[['Story', 'OutputCase','CaseType','Direction','Drift','Label','X','Y','Z']].copy() #extraigo lo que necesito 
#
#df_drifts['Drift'] = df_drifts['Drift'].astype(float) # convertimos de string a float
#
#
#orden_pisos = df_drifts['Story'].unique()
#df_drifts['Story'] = pd.Categorical(df_drifts['Story'], categories=orden_pisos, ordered=True) # le indicamos el orden de los pisos 
#drifts_max = df_drifts.groupby(['Story','Direction'], observed=False, sort=False)['Drift'].max().reset_index()  # buscamos el mayor de los valores para cada piso y direccion
##Reorganizamos el df
#df_resum_drift = drifts_max.pivot(index='Story',columns ='Direction',values='Drift').reset_index() # reorganizamos la serie, columnas de X y Y
#
#df_resum_drift.columns.name = None #eliminamos el nombre del indique de las columnas (eliminamos 'Direction')
##df_resum_drift = df_resum_drift[['Story', 'X', 'Y']]
##print(df_resum_drift)
#
## Organizamo orden----
#hoja.range(f'A{fila_inicio+num_stories+395}').options(pd.DataFrame, index=False, header=False).value = df_resum_drift # pego la tabla de modal 
#

,Story,OutputCase,CaseType,StepType,StepNumber,StepLabel,Direction,Drift,Drift/,Label,X,Y,Z
0,cercha 2,U_B.2.4-5_1 DER_H,Combination,Max,None,None,X,0.004989,1/200,735,32.1706,-4.5445,8.16
1,cercha 2,U_B.2.4-5_1 DER_H,Combination,Max,None,None,Y,0.001437,1/696,750,31.2696,0.6562,8.16
2,cercha 2,U_B.2.4-5_1 DER_H,Combination,Min,None,None,X,0.00588,1/170,738,32.1706,-1.2445,8.16
3,cercha 2,U_B.2.4-5_1 DER_H,Combination,Min,None,None,Y,0.002764,1/362,750,31.2696,0.6562,8.16
4,cercha 2,U_B.2.4-5_2 DER_H,Combination,Max,None,None,X,0.004989,1/200,735,32.1706,-4.5445,8.16


In [56]:
Stiffness_tabla = "Story Stiffness" # Nombre de la tabla donde se extrae la info

#Extraemos la tabla completa
version_tabla_Stif= 0 # version de la tabla 
nombres_colums_Stif = [] # guardamos el nombre de las columnas
num_filas_Stif= 0 # guardamos el numero de filas
TableData_Stif= [] # informacion completa de la tabla drifts de etabs


etabs_model.DatabaseTables.SetLoadCasesSelectedForDisplay(['FHEX','FHEY']) # LE INDICO CASO QUE QUIERO IMPORTAR 


#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, version_tabla_Stif, nombres_colums_Stif, num_filas_Stif, TableData_Stif, ret = etabs_model.DatabaseTables.GetTableForDisplayArray( Stiffness_tabla, nombres_colums_Stif,"", version_tabla_Stif, nombres_colums_Stif, num_filas_Stif, TableData_Stif )
#print(nombres_colums_modal)   # nombres de columnas            #################### PODRIA SOLO IMPORTAR OUTCASE Y UX ####################


n_cols = len(nombres_colums_Stif) # cantidad de columnas extraidas de la tabla
array_Stif= np.array(TableData_Stif).reshape(-1, n_cols)  # convertimos la TableData_modal en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
Stif_tabla = pd.DataFrame(array_Stif, columns=nombres_colums_Stif) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

Stif_tabla.head() # muestra una parte de la tabla.

### Extramos la masa por piso
df_Stif = Stif_tabla[['Story', 'OutputCase','ShearX','DriftX','StiffX','ShearY','DriftY','StiffY']].copy() #extraigo lo que necesito 
#
Stif_FHEX = df_Stif[df_Stif['OutputCase'] == 'FHEX']  # df_Stif['OutputCase'] == 'FHEX' -> busca donde hay registro de FHEX en OutputCase...
Stif_FHEY = df_Stif[df_Stif['OutputCase'] == 'FHEY']  # Stif_FHEX = df_Stif[df_BR... --> Extrae las fila de informacion completa donde hay la coincidencia. 

Stif_FHEX = Stif_FHEX[['Story','OutputCase','ShearX','DriftX','StiffX']]
Stif_FHEY = Stif_FHEY[['OutputCase','ShearY','DriftY','StiffY']]

#print(Stif_FHEX)
#print(Stif_FHEY)

hoja.range(f'A{fila_inicio+num_stories+448}').options(pd.DataFrame, index=False, header=False).value = Stif_FHEX # pego la tabla
hoja.range(f'F{fila_inicio+num_stories+448}').options(pd.DataFrame, index=False, header=False).value = Stif_FHEY # pego la tabla



# CORTANTE DE PISO - AUTO LATERAL LOADS 

In [57]:
############################################ AUTO LATERAL LOADS --  STORY FORCES #################################################################

Lateral_tabla = "Story Forces" # Nombre de la tabla donde se extrae la info

#Extraemos la tabla completa
version_tabla_lateral= 0 # version de la tabla 
nombres_colums_lateral = [] # guardamos el nombre de las columnas
num_filas_lateral= 0 # guardamos el numero de filas
TableData_lateral= [] # informacion completa de la tabla drifts de etabs

etabs_model.DatabaseTables.SetLoadCasesSelectedForDisplay(['FHEX']) # LE INDICO CASO QUE QUIERO IMPORTAR 
etabs_model.DatabaseTables.SetLoadCombinationsSelectedForDisplay([])

#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, version_tabla_lateral, nombres_colums_lateral, num_filas_lateral, TableData_lateral, ret = etabs_model.DatabaseTables.GetTableForDisplayArray( Lateral_tabla, nombres_colums_lateral,"", version_tabla_lateral, nombres_colums_lateral, num_filas_lateral, TableData_lateral )

n_cols = len(nombres_colums_lateral) # cantidad de columnas extraidas de la tabla
array_lateral= np.array(TableData_lateral).reshape(-1, n_cols)  # convertimos la TableData_modal en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
lateral_tabla = pd.DataFrame(array_lateral, columns=nombres_colums_lateral) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

lateral_tabla.head() # muestra una parte de la tabla.


df_lateral = lateral_tabla[['Story','OutputCase','VX','Location']].copy() #extraigo lo que necesito 

lateral_FHEX = df_lateral[(df_lateral['OutputCase']=='FHEX') & (df_lateral['Location']=='Top')]

lateral_FHEX = lateral_FHEX[['Story','VX']]
print(lateral_FHEX)

### importamo la informacion de elevacion del modelo
#[_,NumberRecords_lv, nombres_colums_lv,num_pisos, TableData_lv, ret] = etabs_model.DatabaseTables.GetTableForDisplayArray( "Story Definitions", [], "", 0, [], 0, [])
num_stor, story_names, story_elevations,_,_,_,_,_,ret = ( etabs_model.Story.GetStories())

df_elevaciones = pd.DataFrame( {"Story": story_names, "Elevation": story_elevations})
df_elevaciones = df_elevaciones.iloc[::-1].reset_index(drop=True)
df_elevaciones =df_elevaciones[['Elevation']]


hoja.range(f'A{fila_inicio+num_stories+478}').options(pd.DataFrame, index=False, header=False).value = lateral_FHEX  # pego la tabla
hoja.range(f'D{fila_inicio+num_stories+478}').options(pd.DataFrame, index=False, header=False).value = df_elevaciones  # pego la tabla


      Story       VX
0  cercha 2  -0.6567
